# Tiny GPT Decoder 10 (Generation Decoding - Top P Sampling)

In the 10th version we are focusing on inference using top P sampling. Top P refers to the cumulative probabilities.

In [1]:
import numpy as np
import scipy.stats as stats
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## LLM Architecture

At high level, the training process is as follows:

```text

Text_processing -> Transformer Network -> Compare Loss -> Back Propagation
```

The inference process is as follows:

```text
Text_processing -> Transformer Network -> Predict Next Token -> Repeat until token size reached
```

### Corpus

In [3]:
# Text corpus
corpus = """
mary had a little lamb
its fleece was white as snow
and everywhere that mary went
the lamb was sure to go
mary had a little lamb
the lamb was sure to go
"""


### Parameters

In [4]:
# Set Parameters

# random seed
random_seed = 42
torch.manual_seed(random_seed)

# d_model is embedding dimension
d_model = 4

# block_size is the same as context window or maximum sequence length
block_size = 5



## Text Processing and Tokenization

In [5]:
# Tokenization

def tiny_tokenization(corpus_or_prompt):
  return corpus_or_prompt.split()

# Split corpus into words
word_tokens = tiny_tokenization(corpus)

# Use set to isolate unique words
vocab = sorted(list(set(word_tokens)))
vocab_size = len(vocab)

# Create token id and create lookup table from word to token id
# i is the index or tken id in this case, so we assign token id using sequential number
words_to_id_ref = {word: i for i, word in enumerate(vocab)}

# Create lookup table from id to words
id_to_words_ref = {i: word for i, word in enumerate(vocab)}

In [6]:
vocab_size

19

In [7]:
# convert corpus to id
corpus_in_id = [words_to_id_ref[word] for word in word_tokens]


## Class and Function Setup

In [8]:
def get_sequence_block(block_size, start_position, corpus_in_id):
  """
  This function extract the block in id from the corpus in id
  User need to supply the block size and the start position
  Return: X, Y, next_start_position
  """
  X = corpus_in_id[start_position: start_position+block_size]
  Y = corpus_in_id[start_position+1: start_position+block_size+1]
  return X, Y

In [9]:
def get_N(corpus_in_id, block_size):
  """
  This function calculate the number of block we need to run for the entire corpus
  """
  N = (len(corpus_in_id)-1) // block_size
  return N

In [10]:
class TinyGPT(nn.Module):

    def __init__(self, vocab_size, d_model):
        super().__init__()

        # Initialize trainable components here
        self.embeddings = nn.Embedding(vocab_size, d_model)
        self.WQ = nn.Parameter(torch.randn(d_model, d_model))
        self.WK = nn.Parameter(torch.randn(d_model, d_model))
        self.WV = nn.Parameter(torch.randn(d_model, d_model))

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model, bias=False),
            nn.ReLU(),
            nn.Linear(d_model, d_model, bias=False)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, token_ids):

        # Perform Transformer calculations here
        # Word embeddings + positional encoding
        X_input = self.embeddings(token_ids)
        position = torch.arange(
            len(token_ids),
            dtype=X_input.dtype,
            device=X_input.device
        ).reshape(-1, 1)
        X = X_input + position

        # Q, K, V projections
        Q = X @ self.WQ
        K = X @ self.WK
        V = X @ self.WV

        # Scaled dot-product attention
        attention_score = Q @ K.T
        stable_factor = self.WQ.shape[-1] ** 0.5
        attention_stable = attention_score / stable_factor

        # Causal masking + softmax
        causal_mask = torch.triu(
            torch.full_like(attention_stable, float("-inf")),
            diagonal=1
        )
        attention_stable_mask = attention_stable + causal_mask
        attention_softmax = torch.softmax(attention_stable_mask, dim=-1)

        # Attention output + residual + norm1
        attention_output = attention_softmax @ V

        # Residual Layer
        attention_output_residual = attention_output + X

        # norm1 layer
        attention_output_residual_scaled = self.norm1(attention_output_residual)

        # FFN + residual + norm2
        ffn_1 = self.ffn(attention_output_residual_scaled)

        # Residual 2 Layer
        seond_residual = ffn_1 + attention_output_residual_scaled

        # Norm 2 Layer
        transformer_output = self.norm2(seond_residual)

        # Vocabulary projection
        logits = transformer_output @ self.embeddings.weight.T

        return logits

### Training Setup

In [11]:
def llm_train(corpus_in_id, block_size, model, criterion, optimizer, epoch=10):

  # Get N
  N = get_N(corpus_in_id, block_size=block_size)

  for i in range(epoch):

    print(f"epoch {i}")

    for j in range(N):

      # Get block
      X_id, Y_id = get_sequence_block(
          block_size=block_size,
          start_position=j * block_size,
          corpus_in_id=corpus_in_id
      )
      X_tensor = torch.tensor(X_id, dtype=torch.long, device=device)
      Y_tensor = torch.tensor(Y_id, dtype=torch.long, device=device)

      optimizer.zero_grad()

      logits = model(X_tensor)
      loss = criterion(logits, Y_tensor)
      print(f"Block {j} : Loss = {loss.item():.4f}")

      loss.backward()
      optimizer.step()

### Train the Entire Corpus

In [12]:
model = TinyGPT(vocab_size=vocab_size, d_model=d_model).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [13]:
llm_train(
    corpus_in_id=corpus_in_id,
    block_size=block_size,
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    epoch=5000
)

Streaming output truncated to the last 5000 lines.
Block 4 : Loss = 0.0085
Block 5 : Loss = 0.0074
epoch 4286
Block 0 : Loss = 0.0054
Block 1 : Loss = 0.0105
Block 2 : Loss = 0.0127
Block 3 : Loss = 0.0143
Block 4 : Loss = 0.0084
Block 5 : Loss = 0.0074
epoch 4287
Block 0 : Loss = 0.0055
Block 1 : Loss = 0.0105
Block 2 : Loss = 0.0128
Block 3 : Loss = 0.0143
Block 4 : Loss = 0.0086
Block 5 : Loss = 0.0074
epoch 4288
Block 0 : Loss = 0.0056
Block 1 : Loss = 0.0105
Block 2 : Loss = 0.0127
Block 3 : Loss = 0.0143
Block 4 : Loss = 0.0083
Block 5 : Loss = 0.0074
epoch 4289
Block 0 : Loss = 0.0055
Block 1 : Loss = 0.0105
Block 2 : Loss = 0.0128
Block 3 : Loss = 0.0143
Block 4 : Loss = 0.0087
Block 5 : Loss = 0.0074
epoch 4290
Block 0 : Loss = 0.0056
Block 1 : Loss = 0.0105
Block 2 : Loss = 0.0127
Block 3 : Loss = 0.0143
Block 4 : Loss = 0.0084
Block 5 : Loss = 0.0074
epoch 4291
Block 0 : Loss = 0.0055
Block 1 : Loss = 0.0105
Block 2 : Loss = 0.0126
Block 3 : Loss = 0.0143
Block 4 : Loss = 0.

We intentionally increase the epoch to get 100% accuracy. This is for model evaluation and inference later.

In [14]:
def evaluate_blocks(model, criterion, corpus_in_id, block_size, N):
    model.eval()

    with torch.no_grad():
        for i in range(N):
            start = i * block_size

            X_id, Y_id = get_sequence_block(
                block_size=block_size,
                start_position=start,
                corpus_in_id=corpus_in_id
            )

            X = torch.tensor(X_id, dtype=torch.long, device=device)
            Y = torch.tensor(Y_id, dtype=torch.long, device=device)

            logits = model(X)
            loss = criterion(logits, Y)

            print(f"Block {i}: Loss = {loss.item():.4f}")

            predicted_ids = torch.argmax(logits, dim=-1)
            correct = (predicted_ids == Y).sum().item()
            accuracy = correct / len(Y)

            print(f"Block {i}: Accuracy = {accuracy:.4f}")

            print("-" * 50)
            print(f"Block {i}")
            for k in range(5):
              print(f"Position {k}, expected = {id_to_words_ref[int(Y[k])]}, predicted = {id_to_words_ref[int(predicted_ids[k])]}")
            print("-" * 50)

In [15]:
N = get_N(corpus_in_id, block_size=block_size)


In [16]:
evaluate_blocks(model=model, criterion=criterion, corpus_in_id=corpus_in_id, block_size=block_size, N=N)

Block 0: Loss = 0.0040
Block 0: Accuracy = 1.0000
--------------------------------------------------
Block 0
Position 0, expected = had, predicted = had
Position 1, expected = a, predicted = a
Position 2, expected = little, predicted = little
Position 3, expected = lamb, predicted = lamb
Position 4, expected = its, predicted = its
--------------------------------------------------
Block 1: Loss = 0.0080
Block 1: Accuracy = 1.0000
--------------------------------------------------
Block 1
Position 0, expected = fleece, predicted = fleece
Position 1, expected = was, predicted = was
Position 2, expected = white, predicted = white
Position 3, expected = as, predicted = as
Position 4, expected = snow, predicted = snow
--------------------------------------------------
Block 2: Loss = 0.0107
Block 2: Accuracy = 1.0000
--------------------------------------------------
Block 2
Position 0, expected = and, predicted = and
Position 1, expected = everywhere, predicted = everywhere
Position 2, exp

## Inference Decoding - Top P

During inference, there are many decoding method:

- Greedy
- Sampling
- Temperature Sampling
- Top K Sampling
- Top P Sampling
- Beam Search

In this exercise, we will be looking at top P.

For Top P, we decide P as the cumulative proabbilities. this way we can eliminate words with lower probabilities.

Consider the example below:

```text
Top 10 probabilities
had, prob=0.99792099
fleece, prob=0.00088941
go, prob=0.00088901
white, prob=0.00023128
snow, prob=0.00005766
as, prob=0.00001151
lamb, prob=0.00000011
was, prob=0.00000004
went, prob=0.00000001
a, prob=0.00000000
```

if our top P is 95% or 0.95, then only teh first word can be selected. So in our exercise we will increase tau to get varielty.

The process is similar to top K. The process is as follows:
Sort probabilities in descending order.
- Calculate cumulative probabilities using torch.cumsum().
- Determine how many words are needed to reach the threshold p.
- Renormalize the selected probabilities.
- Sample using torch.multinomial().
- Map the sampled position back to the original vocabulary ID.

### Functions for Inference

In [17]:
def prompt_processing(user_prompt_text):
  user_prompt_text = user_prompt_text.split()
  prompt_id = [words_to_id_ref[each] for each in user_prompt_text]
  return torch.tensor(prompt_id, dtype=torch.long, device=device)

### Inference 1 - Top P Sampling

In [165]:
def prompt_inference_topp(user_prompt, max_token, model, tau, p, id_to_words_dict):
  X = prompt_processing(user_prompt)
  for i in range(max_token):
    model.eval()

    # the code below prevent torch for tracking the gradient since we are inferencing and not training
    with torch.no_grad():
        prompt_logits = model(X)

    # predicting token after
    token = [id_to_words_dict[each.item()] for each in X]
    print(f"Predicting: last token of the prompt {token}")

    # Set the temperature for the last logits
    predict_logits_before_temperature = prompt_logits[-1]
    predict_logits = predict_logits_before_temperature / tau

    # get the last token prediction
    next_token_probs = torch.softmax(predict_logits, dim=-1)

    # select top p
    sorted_probs, sorted_indices = torch.sort(next_token_probs, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=0)
    num_selected = torch.sum(cumulative_probs < p).item() + 1
    print(f"Cumulative probability: {cumulative_probs[num_selected - 1].item():.8f}")

    # display original prob distribution
    print("------")
    print(f"Original Prob distribution")
    for word, prob in zip(sorted_indices[:5], sorted_probs[:5]):
      print(f"{id_to_words_dict[word.item()]}, prob={prob.item():.8f}")
    print("------")
    print(f"Number of top P selected {num_selected}")

    # The code below prevent the num_selected to be more than vocab size
    num_p = min(num_selected, len(sorted_probs))

    selected_token_ID = sorted_indices[:num_p]
    selected_probs = sorted_probs[:num_p]

    # normalized logits
    selected_logits_norm = selected_probs / torch.sum(selected_probs)

    # print the selected pool
    print("------")
    print(f"Selected word -> ", end='')
    for word in selected_token_ID:
      print(f"{id_to_words_dict[word.item()]}, ", end='')
    print("\n")
    print("===========")

    # next token prediction - sampling
    next_token_id = torch.multinomial(selected_logits_norm, num_samples=1)

    X = torch.cat([X, selected_token_ID[next_token_id]])
  return X

In [166]:
def generate_response(X, id_to_words_dict):
  response = [id_to_words_dict[each.item()] for each in X]
  print(f"Prompt: {user_prompt}")
  print(f"Response: {' '.join(response)}")

In [167]:
user_prompt = "mary"
user_prompt

'mary'

In [168]:
max_token = 4
tau = 1
p = 0.9

In [169]:
X = prompt_inference_topp(user_prompt, max_token, model, tau, p, id_to_words_ref)

Predicting: last token of the prompt ['mary']
Cumulative probability: 0.99792093
------
Original Prob distribution
had, prob=0.99792093
fleece, prob=0.00088941
go, prob=0.00088901
white, prob=0.00023128
snow, prob=0.00005766
------
Number of top P selected 1
------
Selected word -> had, 

Predicting: last token of the prompt ['mary', 'had']
Cumulative probability: 0.99519891
------
Original Prob distribution
a, prob=0.99519891
and, prob=0.00454428
go, prob=0.00021243
went, prob=0.00002480
little, prob=0.00001886
------
Number of top P selected 1
------
Selected word -> a, 

Predicting: last token of the prompt ['mary', 'had', 'a']
Cumulative probability: 0.99511969
------
Original Prob distribution
little, prob=0.99511969
everywhere, prob=0.00213704
and, prob=0.00182014
that, prob=0.00089693
a, prob=0.00002145
------
Number of top P selected 1
------
Selected word -> little, 

Predicting: last token of the prompt ['mary', 'had', 'a', 'little']
Cumulative probability: 0.99708003
------


In [170]:
generate_response(X, id_to_words_ref)

Prompt: mary
Response: mary had a little lamb


We still get a good prediction. Our softmax is too high for get variety. The following example we will increase temperature.

### Inference 2 - Tau = 5, p = 0.5

In [171]:
user_prompt = "mary"
user_prompt

'mary'

In [172]:
max_token = 4
tau = 5
p = 0.5

In [175]:
X = prompt_inference_topp(user_prompt, max_token, model, tau, p, id_to_words_ref)

Predicting: last token of the prompt ['mary']
Cumulative probability: 0.59562492
------
Original Prob distribution
had, prob=0.47823238
fleece, prob=0.11739253
go, prob=0.11738192
white, prob=0.08966966
snow, prob=0.06792036
------
Number of top P selected 2
------
Selected word -> had, fleece, 

Predicting: last token of the prompt ['mary', 'had']
Cumulative probability: 0.52252746
------
Original Prob distribution
a, prob=0.52252746
and, prob=0.17783621
go, prob=0.09637659
went, prob=0.06272005
little, prob=0.05937656
------
Number of top P selected 1
------
Selected word -> a, 

Predicting: last token of the prompt ['mary', 'had', 'a']
Cumulative probability: 0.58519256
------
Original Prob distribution
little, prob=0.45269850
everywhere, prob=0.13249406
and, prob=0.12830830
that, prob=0.11137449
a, prob=0.05278420
------
Number of top P selected 2
------
Selected word -> little, everywhere, 

Predicting: last token of the prompt ['mary', 'had', 'a', 'everywhere']
Cumulative probabi

In [176]:
generate_response(X, id_to_words_ref)

Prompt: mary
Response: mary had a everywhere that


If you still get a correct response, try running a few time, it will change

### Inference 3 -  Tau = 5, p = 0.8

In [185]:
user_prompt = "mary had a"
user_prompt

'mary had a'

In [186]:
max_token = 2
tau = 5
p = 0.8

In [187]:
X = prompt_inference_topp(user_prompt, max_token, model, tau, p, id_to_words_ref)

Predicting: last token of the prompt ['mary', 'had', 'a']
Cumulative probability: 0.82487535
------
Original Prob distribution
little, prob=0.45269850
everywhere, prob=0.13249406
and, prob=0.12830830
that, prob=0.11137449
a, prob=0.05278420
------
Number of top P selected 4
------
Selected word -> little, everywhere, and, that, 

Predicting: last token of the prompt ['mary', 'had', 'a', 'everywhere']
Cumulative probability: 0.86130488
------
Original Prob distribution
mary, prob=0.31486905
that, prob=0.24088830
went, prob=0.20792821
little, prob=0.09761928
snow, prob=0.03989739
------
Number of top P selected 4
------
Selected word -> mary, that, went, little, 



In [188]:
generate_response(X, id_to_words_ref)


Prompt: mary had a
Response: mary had a everywhere that


Sampling is probabilistic, not deterministic. The same output is possible, but not guaranteed.

## END